## Import Libraries and Some Function

In [1]:
import pandas as pd
import numpy as np
import random
import requests
import time
import re
from urllib.parse import urljoin

from bs4 import BeautifulSoup

In [2]:
BASE_URL = "https://www.vlr.gg"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/142.0")
}

In [3]:
def absolute(base_url: str = BASE_URL, url: str = None):
    return urljoin(base=base_url, url=url)

In [4]:
def get_soup(
        url: str,
        headers: dict = None,
        retry: int = 3,
        delay_range: set = (5, 10)) -> None:
    session = requests.session()
    session.headers.update(headers=headers)
    for attempt in range(retry):
        try:
                response = session.get(url=url, timeout=10)
                response.raise_for_status()
                if response.status_code == 429:
                    time.sleep(300)
                    continue

                time.sleep(random.uniform(25, 30))
                return BeautifulSoup(response.content, "html.parser")
        except Exception as e:
            print(f"Attempt {attempt + 1} Failed: {e}")
            time.sleep(random.uniform(*delay_range))
    
    print(f"failed to get soup for {url} after {retry} attempts.")
    return None
    

## Getting Tournaments Information

**Getting List of Tournaments**

In [5]:
soup = get_soup(url=BASE_URL)
contents = soup.select(".header-inner a[href]")
for content in contents:
    href = content.get("href")
    if "events" not in href:
        continue
    url = absolute(url=href)

soup = get_soup(url=url)
contents = soup.select(".wf-filter-inner a[href]")
for content in contents:
    href = content.get("href")
    if "60" not in href:
        continue
    url = absolute(url=href)

soup = get_soup(url=url)
contents = soup.select(".events-container-col a[href]")
results = set()
for content in contents:
    href = content.get("href")
    url = absolute(url=href)
    results.add(url)

results = list(results)
results[:5]

['https://www.vlr.gg/event/2953/esports-world-cup-2026-americas-qualifier',
 'https://www.vlr.gg/event/2684/vct-2026-emea-kickoff',
 'https://www.vlr.gg/event/2277/vct-2025-pacific-kickoff',
 'https://www.vlr.gg/event/2275/vct-2025-china-kickoff',
 'https://www.vlr.gg/event/2339/china-evolution-series-act-1']

**Scraping Tournament Details**

In [6]:
tournaments_dict = {
    "tour_id": [],
    "tour_name": [],
    "tour_tag": [],
    "tour_stage": [],
    "tour_region": []
}

In [7]:
url = "https://www.vlr.gg/event/2683/vct-2026-pacific-kickoff"
tournament_id = url.split("/")[4]
soup = get_soup(url=url)
title_container = soup.select_one(".wf-title")
title = title_container.get_text().strip()
desc = soup.select(".event-desc-inner a[href]")
tour = desc[0].get_text().strip()
# Add an if condition for tour_tag bypass
stage = desc[1].get_text().strip()
region = desc[2].get_text().strip()
print(f"This is a {stage} stage on {tour} in {region} region.")
tournaments_dict["tour_id"].append(tournament_id)
tournaments_dict["tour_name"].append(title)
tournaments_dict["tour_tag"].append(tour)
tournaments_dict["tour_stage"].append(stage)
tournaments_dict["tour_region"].append(region)
contents = soup.select(".wf-nav a[href]")
matches_page = set()
stats_page = set()
agents_page = set()
for content in contents:
    href = content.get("href")
    url = absolute(url=href)
    if "matches" in href:
        matches_page.add((tournament_id, url))
    elif "stats" in href:
        stats_page.add((tournament_id, url))
    elif "agents" in href:
        agents_page.add((tournament_id, url))
    else:
        continue

for variable in [matches_page, stats_page, agents_page]:
    variable = list(variable)
    print(f"Url exists in {variable}" if len(variable) > 0 else f"Url not exists in {variable}")

tournaments_df = pd.DataFrame(tournaments_dict)
tournaments_df

This is a Kickoff stage on Valorant Champions Tour 2026 in Pacific region.
Url exists in [('2683', 'https://www.vlr.gg/event/matches/2683/vct-2026-pacific-kickoff/?series_id=5222')]
Url exists in [('2683', 'https://www.vlr.gg/event/stats/2683/vct-2026-pacific-kickoff')]
Url exists in [('2683', 'https://www.vlr.gg/event/agents/2683/vct-2026-pacific-kickoff')]


,tour_id,tour_name,tour_tag,tour_stage,tour_region
0,2683,VCT 2026: Pacific Kickoff,Valorant Champions Tour 2026,Kickoff,Pacific


## Getting Matches Information

In [ ]:
matches_dict = {
    "tour_id": [],
    "match_id": [],
    "bracket": [],
    "date": [],
    "patch": [],
    "home_team": [],
    "home_alias": [],
    "away_team": [],
    "away_alias": [],
    "home_score": [],
    "away_score": [],
    "bo": [],
    "home_map_ban_1": [],
    "home_map_ban_2": [],
    "home_map_ban_3": [],
    "home_map_pick_1": [],
    "home_map_pick_2": [],
    "away_map_ban_1": [],
    "away_map_ban_2": [],
    "away_map_ban_3": [],
    "away_map_pick_1": [],
    "away_map_pick_2": [],
    "decider_map": [],
    "home_h2h_win": [],
    "away_h2h_win": [],
    "home_h2h_score": [],
    "away_h2h_score": [],
    "home_last_5_wr": [],
    "away_last_5_wr": []
}

**Getting List of Match**

In [ ]:
# Replace series_id with "all"
page = [["2683", "https://www.vlr.gg/event/matches/2683/vct-2026-pacific-kickoff/?series_id=all"]]
tour_id, url  = page[0]
soup = get_soup(url=url)
match_containers = soup.select(".wf-card a[href]")
matches = set()
for match in match_containers:
    href = match.get("href")
    match_id = href.split("/")[1]
    if "5956" not in href:
        continue
    url = absolute(url=href)
    matches.add(url)

result = list(matches)
result[:5]

['https://www.vlr.gg/595645/detonation-focusme-vs-global-esports-vct-2026-pacific-kickoff-mr2',
 'https://www.vlr.gg/595647/full-sense-vs-detonation-focusme-vct-2026-pacific-kickoff-mr3',
 'https://www.vlr.gg/595652/varrel-vs-gen-g-vct-2026-pacific-kickoff-lr1',
 'https://www.vlr.gg/595639/paper-rex-vs-rex-regum-qeon-vct-2026-pacific-kickoff-ur3',
 'https://www.vlr.gg/595637/rex-regum-qeon-vs-detonation-focusme-vct-2026-pacific-kickoff-ur2']

**Scraping Match Details**

In [ ]:
url = "https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1"
soup = get_soup(url=url)
bracket_info_container = soup.select_one(".match-header-event-series")
bracket_name = bracket_info_container.get_text().split(":")[1].strip()
match_id_container = soup.select_one(".vm-stats-tabnav a")
match_id = match_id_container.get("data-match-id")
date_info_container = soup.select_one(".moment-tz-convert")
date = date_info_container.get("data-utc-ts")
patch_info_container = soup.select_one(".match-header-date")
patch = re.sub(r"[\n\t]", "", patch_info_container.get_text()).split("Patch")[-1].strip()

matches_dict["tour_id"].append(tour_id)
matches_dict["match_id"].append(match_id)
matches_dict["date"].append(date)
matches_dict["bracket"].append(bracket_name)
matches_dict["patch"].append(patch)

print(f"Match ID -> {match_id}")
print(f"Match date -> {date}")
print(f"Match bracket -> {bracket_name}")
print(f"Patch info -> {patch}")

Match_id -> 595630
Match date -> 2026-01-22 03:00:00
Match bracket -> Upper Round 1
Patch info -> 12.0


In [23]:
home_team_container = soup.select_one(".match-header-link.wf-link-hover.mod-1")
home_href = home_team_container.get("href")
home_url = absolute(url=home_href)
home_soup = get_soup(home_url)
home_container = home_soup.select('.wf-title')
home_name = home_container[0].get_text()
home_alias = home_container[1].get_text()

away_team_container = soup.select_one(".match-header-link.wf-link-hover.mod-2")
away_href = away_team_container.get("href")
away_url = absolute(url=away_href)
away_soup = get_soup(away_url)
away_container = away_soup.select('.wf-title')
away_name = away_container[0].get_text()
away_alias = away_container[1].get_text()

bo_info_container = soup.select(".match-header-vs-note")
bo_info = bo_info_container[1].get_text().strip()
score_container = soup.select_one(".js-spoiler")
home_score = re.sub(r"[\n\t]", "", score_container.get_text()).split(":")[0].strip()
away_score = re.sub(r"[\n\t]", "", score_container.get_text()).split(":")[-1].strip()

matches_dict["home_team"].append(home_name)
matches_dict["home_alias"].append(home_alias)
matches_dict["away_team"].append(away_name)
matches_dict["away_alias"].append(away_alias)
matches_dict["home_score"].append(home_score)
matches_dict["away_score"].append(away_score)
matches_dict["bo"].append(bo_info)

print(f"match -> {home_name} ({home_alias}) vs ({away_alias}) {away_name}")
print(f"score -> {home_score} : {away_score}")
print(f"BO -> {bo_info}")

match -> Nongshim RedForce (NS) vs (TS) Team Secret
score -> 2 : 0
BO -> Bo3


In [ ]:
map_selection_container = soup.select_one(".match-header-note")
if not map_selection_container:
    for key in matches_dict.keys():
        matches_dict[key].append(np.nan)
    print("Container does not exist")
map_selection = map_selection_container.get_text().split(";")
home_map_picks, home_map_bans = [], []
away_map_picks, away_map_bans = [], []
decider_map = ""
for map in map_selection:
    map_split = map.strip().split(" ")
    if map_split[0] == home_alias:
        if map_split[1] == "pick":
            home_map_picks.append(map_split[2].strip())
        else:
            home_map_bans.append(map_split[2].strip())
    elif map_split[0] == away_alias:
        if map_split[1] == "pick":
            away_map_picks.append(map_split[2].strip())
        else:
            away_map_bans.append(map_split[2].strip())
    else:
        decider_map = map_split[0].strip()

matches_dict["home_map_ban_1"].append(home_map_bans[0])
matches_dict["home_map_pick_1"].append(home_map_picks[0])
matches_dict["away_map_ban_1"].append(away_map_bans[0])
matches_dict["away_map_pick_1"].append(away_map_picks[0])
if bo_info == "Bo3":
    matches_dict["home_map_ban_2"].append(home_map_bans[1])
    matches_dict["home_map_ban_3"].append(np.nan)
    matches_dict["home_map_pick_2"].append(np.nan)
    matches_dict["away_map_ban_2"].append(away_map_bans[1])
    matches_dict["away_map_ban_3"].append(np.nan)
    matches_dict["away_map_pick_2"].append(np.nan)
elif bo_info == "Bo5":
    matches_dict["home_map_ban_2"].append(np.nan)
    matches_dict["home_map_ban_3"].append(np.nan)
    matches_dict["home_map_pick_2"].append(home_map_picks[1])
    matches_dict["away_map_ban_2"].append(np.nan)
    matches_dict["away_map_ban_3"].append(np.nan)
    matches_dict["away_map_pick_2"].append(away_map_picks[1])
matches_dict["decider_map"].append(decider_map)

In [25]:
head_to_head_container = soup.select(".match-h2h-matches-score")
home_h2h_win = 0
away_h2h_win = 0
home_h2h_score = 0
away_h2h_score = 0
for container in head_to_head_container:
    score = container.get_text().strip().split("\n")
    home_h2h_score += int(score[0])
    away_h2h_score += int(score[1])
    if int(score[0]) > int(score[1]):
        home_h2h_win += 1
    else:
        away_h2h_win += 1

matches_dict["home_h2h_win"].append(home_h2h_win)
matches_dict["away_h2h_win"].append(away_h2h_win)
matches_dict["home_h2h_score"].append(home_h2h_score)
matches_dict["away_h2h_score"].append(away_h2h_score)

print(f"head-to-head win: Home {home_h2h_win} vs {away_h2h_win} Away")
print(f"head-to-head score: Home {home_h2h_score} vs {away_h2h_score} Away")

head-to-head win: Home 2 vs 1 Away
head-to-head score: Home 4 vs 2 Away


In [26]:
last_match_container = soup.select(".wf-card.mod-dark.match-histories")
home_last_match = last_match_container[0].select(".match-histories-item.wf-module-item")
home_last_match_win = 0
for match in home_last_match:
    match_class = match.get("class")
    if "mod-win" not in match_class:
        continue
    home_last_match_win += 1
    home_last_match_wr = home_last_match_win / 5

away_last_match = last_match_container[1].select(".match-histories-item.wf-module-item")
away_last_match_win = 0
for match in away_last_match:
    match_class = match.get("class")
    if "mod-win" not in match_class:
        continue
    away_last_match_win += 1
    away_last_match_wr = away_last_match_win / 5

matches_dict["home_last_5_wr"].append(home_last_match_wr)
matches_dict["away_last_5_wr"].append(away_last_match_wr)

print(f"Home winrate last 5 matches: {home_last_match_wr}")
print(f"Away winrate last 5 matches: {away_last_match_wr}")

Home winrate last 5 matches: 0.8
Away winrate last 5 matches: 0.4


In [27]:
matches_df = pd.DataFrame(matches_dict)
matches_df

,tour_id,match_id,bracket,date,patch,home_team,home_alias,away_team,away_alias,home_score,...,away_map_ban_2,away_map_pick_1,away_map_pick_2,decider_map,home_h2h_win,away_h2h_win,home_h2h_score,away_h2h_score,home_last_5_wr,away_last_5_wr
0,2683,595630,Upper Round 1,2026-01-22 03:00:00,12.0,Nongshim RedForce,NS,Team Secret,TS,2,...,Abyss,Haven,NaN,[Breeze],2,1,4,2,0.8,0.4


## Getting Game Information

In [127]:
games_dict = {
    "match_id": [],
    "game_id": [],
    "map": [],
    "duration": [],
    "home_score": [],
    "away_score": [],
    "home_att_score": [],
    "away_att_score": [],
    "home_def_score": [],
    "away_def_score": [],
    "home_att_score": [],
    "away_att_score": [],
    "home_pstl_win": [],
    "away_pstl_win": [],
    "home_eco_round": [],
    "away_eco_round": [],
    "home_eco_win": [],
    "away_eco_win": [],
    "home_semi_eco_round": [],
    "away_semi_eco_round": [],
    "home_semi_eco_win": [],
    "away_semi_eco_win": [],
    "home_semi_buy_round": [],
    "away_semi_buy_round": [],
    "home_semi_buy_win": [],
    "away_semi_buy_win": [],
    "home_full_buy_round": [],
    "away_full_buy_round": [],
    "home_full_buy_win": [],
    "away_full_buy_win": [],
}

In [128]:
url = "https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1/"
soup = get_soup(url=url)

In [129]:
tabnav_container = soup.select(".vm-stats-tabnav a")
tab_info = set()
for tab in tabnav_container:
    data_tab = tab.get("data-tab")
    href = url + f"&tab={data_tab}"
    tab_info.add(href)
result = sorted(list(tab_info))
result


['https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1/&tab=economy',
 'https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1/&tab=overview',
 'https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1/&tab=performance']

**Scraping Game Details**

In [130]:
gamesnav_container = soup.select(".vm-stats-gamesnav-item")
game_id = sorted([i.get("data-game-id") for i in gamesnav_container])
games = soup.select(".vm-stats-game-header")
for i, game in enumerate(games):
    if game_id == "all":
        continue
    score_container = game.select(".score")
    home_score = score_container[0].get_text().strip()
    away_score = score_container[1].get_text().strip()
    map_duration_container = game.select_one(".map")
    map_duration = re.sub(r"[\n\t]", "", map_duration_container.get_text().strip()).split("PICK")
    map_name = map_duration[0]
    duration = map_duration[1]
    att_score_container = game.select(".mod-t")
    home_att_score = att_score_container[0].get_text().strip()
    away_att_score = att_score_container[1].get_text().strip()
    def_score_container = game.select(".mod-ct")
    home_def_score = def_score_container[0].get_text().strip()
    away_def_score = def_score_container[1].get_text().strip()

    games_dict["match_id"].append(match_id)
    games_dict["game_id"].append(game_id[i])
    games_dict["map"].append(map_name)
    games_dict["duration"].append(duration)
    games_dict["home_score"].append(home_score)
    games_dict["away_score"].append(away_score)
    games_dict["home_att_score"].append(home_att_score)
    games_dict["away_att_score"].append(away_att_score)
    games_dict["home_def_score"].append(home_def_score)
    games_dict["away_def_score"].append(away_def_score)
    
    print(f"Game ID -> {game_id[i]}")
    print(f"Score -> Home {home_score} vs {away_score} Away")
    print(f"Map -> {map_name}")
    print(f'Duration -> {duration}')
    print(f"Attack Score -> Home {home_att_score} vs {away_att_score} Away")
    print(f"Defense Score -> Home {home_def_score} vs {away_def_score} Away")
    print("="*50)

Game ID -> 244560
Score -> Home 13 vs 5 Away
Map -> Haven
Duration -> 59:33
Attack Score -> Home 9 vs 2 Away
Defense Score -> Home 4 vs 3 Away
Game ID -> 244561
Score -> Home 13 vs 8 Away
Map -> Split
Duration -> 46:20
Attack Score -> Home 8 vs 7 Away
Defense Score -> Home 5 vs 1 Away


#### Getting Round Details

In [131]:
url = "https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1/?tab=economy"
soup2 = get_soup(url=url)

In [132]:
games = soup2.select(".wf-table-inset.mod-econ")
for i, game in enumerate(games):
    if len(games) == i + 1:
        break
    stats_info = game.select(".stats-sq")
    stats = [re.sub(r"[\n\t]", "", i.get_text().strip()) for i in stats_info]
    if stats == []:
        continue
    clean_stat = []
    for stat in stats:
        if "(" in stat:
            stat_split = stat.split("(")
            round = stat_split[0]
            round_win = stat_split[1].split(")")[0]
            clean_stat.append(round)
            clean_stat.append(round_win)
        else:
            clean_stat.append(stat)
    home_stats, away_stats= clean_stat[:9], clean_stat[9:]

    games_dict["home_pstl_win"].append(home_stats[0])
    games_dict["away_pstl_win"].append(away_stats[0])
    games_dict["home_eco_round"].append(home_stats[1])
    games_dict["away_eco_round"].append(away_stats[1])
    games_dict["home_eco_win"].append(home_stats[2])
    games_dict["away_eco_win"].append(away_stats[2])
    games_dict["home_semi_eco_round"].append(home_stats[3])
    games_dict["away_semi_eco_round"].append(away_stats[3])
    games_dict["home_semi_eco_win"].append(home_stats[4])
    games_dict["away_semi_eco_win"].append(away_stats[4])
    games_dict["home_semi_buy_round"].append(home_stats[5])
    games_dict["away_semi_buy_round"].append(away_stats[5])
    games_dict["home_semi_buy_win"].append(home_stats[6])
    games_dict["away_semi_buy_win"].append(away_stats[6])
    games_dict["home_full_buy_round"].append(home_stats[7])
    games_dict["away_full_buy_round"].append(away_stats[7])
    games_dict["home_full_buy_win"].append(home_stats[8])
    games_dict["away_full_buy_win"].append(away_stats[8])

    if i == 0:
        print(f"Pistol round win: {home_stats[0]}")
        print(f"Eco round win: {home_stats[1]}")
        print(f"Eco round win: {home_stats[2]}")
        print(f"Semi-eco round win: {home_stats[3]}")
        print(f"Semi-eco round win: {home_stats[4]}")
        print(f"Semi-buy round win: {home_stats[5]}")
        print(f"Semi-buy round win: {home_stats[6]}")
        print(f"Full-buy round win: {home_stats[7]}")
        print(f"Full-buy round win: {home_stats[8]}")

Pistol round win: 2
Eco round win: 2
Eco round win: 2
Semi-eco round win: 0
Semi-eco round win: 0
Semi-buy round win: 4
Semi-buy round win: 3
Full-buy round win: 12
Full-buy round win: 8


In [133]:
games_df = pd.DataFrame(games_dict)
games_df
# for key in games_dict.keys():
#     print(len(games_dict[key]))

,match_id,game_id,map,duration,home_score,away_score,home_att_score,away_att_score,home_def_score,away_def_score,...,home_semi_eco_win,away_semi_eco_win,home_semi_buy_round,away_semi_buy_round,home_semi_buy_win,away_semi_buy_win,home_full_buy_round,away_full_buy_round,home_full_buy_win,away_full_buy_win
0,102603,244560,Haven,59:33,13,5,9,2,4,3,...,0,0,4,4,3,4,12,8,8,1
1,102603,244561,Split,46:20,13,8,8,7,5,1,...,1,0,3,5,3,2,13,13,8,5


### Getting player Details

In [113]:
player_stats_dict = {
    "match_id": [],
    "game_id": [],
    "name": [],
    "team": [],
    "nationality": [],
    "stat_scope": [],
    "side": [],
    "r": [],
    "acs": [],
    "k": [],
    "d": [],
    "a": [],
    "k-d": [],
    "kast": [],
    "adr": [],
    "hs%": [],
    "fk": [],
    "fd": [],
    "fk-fd": [],
}

In [114]:
player_agent_dict = {
    "game_id": [],
    "name": [],
    "agent": []
}

In [62]:
url = "https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1"
soup = get_soup(url=url)
tabnav_container = soup.select_one(".vm-stats-tabnav a")
match_id = tabnav_container.get("data-match-id")
gamesnav_container = soup.select(".vm-stats-gamesnav-item")
game_id = [i.get("data-game-id") for i in gamesnav_container]

In [115]:
games = soup.select(".wf-table-inset.mod-overview")
n = 1
for idx, game in enumerate(games):
    plyr_flag_container = game.select(".flag")
    plyr_flag = [i.get("title").strip() for i in plyr_flag_container]
    plyr_name_container = game.select(".mod-player a")
    plyr_name = [re.sub(r"[\n\t]", "", i.get_text().strip().split(" ")[0]) for i in plyr_name_container]
    plyr_team = [re.sub(r"[\n\t]", "", i.get_text().strip().split(" ")[1]) for i in plyr_name_container]
    agents_container = game.select(".mod-agents")
    agents_list = [i.select(".stats-sq.mod-agent img") for i in agents_container]
    if any(len(agent) > 1 for agent in agents_list):
        for side in ["att", "def", "all"]:
            for i in plyr_name:
                player_stats_dict["match_id"].append(match_id)
                player_stats_dict["game_id"].append(np.nan)
                player_stats_dict["name"].append(i)
            for i in plyr_team:
                player_stats_dict["team"].append(i)
            for i in plyr_flag:
                player_stats_dict["nationality"].append(i)
                player_stats_dict["stat_scope"].append("all_map")
        
        continue
    
    agents_name = []
    for agent in agents_list:
        if len(agent) == 1:
            agents_title = agent[0].get("title").strip()
            agents_name.append(agents_title)
        else:
            print("Agent does not exists.")
    
    # Update player dict
    for side in ["att", "def", "all"]:
        for i in plyr_name:
            player_stats_dict["match_id"].append(match_id)
            player_stats_dict["game_id"].append(game_id[n])
            player_stats_dict["name"].append(i)
        for i in plyr_team:
            player_stats_dict["team"].append(i)
        for i in plyr_flag:
            player_stats_dict["nationality"].append(i)
            player_stats_dict["stat_scope"].append("per_map")
    
    for i in plyr_name:
        player_agent_dict["game_id"].append(game_id[n])
        player_agent_dict["name"].append(i)
    for i in agents_name:
        player_agent_dict["agent"].append(i)
    # Result
    if idx < 2 and idx % 2 == 0:
        print(f"Player 1 name [agent] -> {plyr_name[0]} [{agents_name[0]}]")
        print(f"Player 2 name [agent] -> {plyr_name[1]} [{agents_name[1]}]")
        print(f"Player 3 name [agent] -> {plyr_name[2]} [{agents_name[2]}]")
        print(f"Player 4 name [agent] -> {plyr_name[3]} [{agents_name[3]}]")
        print(f"Player 5 name [agent] -> {plyr_name[4]} [{agents_name[4]}]")
        print("="*50)
    
    elif idx < 2 and idx % 2 == 1:
        n += 1
        print(f"Player 1 name [agent] -> {plyr_name[0]} [{agents_name[0]}]")
        print(f"Player 2 name [agent] -> {plyr_name[1]} [{agents_name[1]}]")
        print(f"Player 3 name [agent] -> {plyr_name[2]} [{agents_name[2]}]")
        print(f"Player 4 name [agent] -> {plyr_name[3]} [{agents_name[3]}]")
        print(f"Player 5 name [agent] -> {plyr_name[4]} [{agents_name[4]}]")
        print("="*50)

Player 1 name [agent] -> Dambi [Neon]
Player 2 name [agent] -> Xross [Sova]
Player 3 name [agent] -> Francis [Yoru]
Player 4 name [agent] -> Rb [Omen]
Player 5 name [agent] -> Ivy [Killjoy]
Player 1 name [agent] -> kellyS [Viper]
Player 2 name [agent] -> Sylvan [Omen]
Player 3 name [agent] -> JessieVash [Sova]
Player 4 name [agent] -> BerserX [Killjoy]
Player 5 name [agent] -> TenTen [Neon]


In [116]:
for i, game in enumerate(games):
    for side in ["att", "def", "all"]:
        map_list = ["Haven", "Split"]
         
        if side == "att":
            plyr_side_stt_container = game.select(".mod-t")
        elif side == "def":
            plyr_side_stt_container = game.select(".mod-ct")
        else:
            plyr_side_stt_container = game.select(".mod-both")
        plyr_side_stt = [i.get_text().strip() for i in plyr_side_stt_container]
        plyr1_side_stt = plyr_side_stt[:12]
        plyr2_side_stt = plyr_side_stt[12:24]
        plyr3_side_stt = plyr_side_stt[24:36]
        plyr4_side_stt = plyr_side_stt[36:48]
        plyr5_side_stt = plyr_side_stt[48:]

        for stat in [plyr1_side_stt, plyr2_side_stt, plyr3_side_stt, plyr4_side_stt, plyr5_side_stt]:
            player_stats_dict["side"].append(side)
            player_stats_dict["r"].append(stat[0])
            player_stats_dict["acs"].append(stat[1])
            player_stats_dict["k"].append(stat[2])
            player_stats_dict["d"].append(stat[3])
            player_stats_dict["a"].append(stat[4])
            player_stats_dict["k-d"].append(stat[5])
            player_stats_dict["kast"].append(stat[6])
            player_stats_dict["adr"].append(stat[7])
            player_stats_dict["hs%"].append(stat[8])
            player_stats_dict["fk"].append(stat[9])
            player_stats_dict["fd"].append(stat[10])
            player_stats_dict["fk-fd"].append(stat[11])

        if i == 0 and side == "att":
            print(f"Rating -> {plyr1_side_stt[0]}")
            print(f"ACS -> {plyr1_side_stt[1]}")
            print(f"Kills -> {plyr1_side_stt[2]}")
            print(f"Deaths -> {plyr1_side_stt[3]}")
            print(f"Assists -> {plyr1_side_stt[4]}")
            print(f"Kill - Death -> {plyr1_side_stt[5]}")
            print(f"KAST -> {plyr1_side_stt[6]}")
            print(f"ADR -> {plyr1_side_stt[7]}")
            print(f"HS% -> {plyr1_side_stt[8]}")
            print(f"First Kill (FK) -> {plyr1_side_stt[9]}")
            print(f"First Death (FD) -> {plyr1_side_stt[10]}")
            print(f"FK - FD -> {plyr1_side_stt[11]}")

Rating -> 1.93
ACS -> 331
Kills -> 15
Deaths -> 6
Assists -> 2
Kill - Death -> +9
KAST -> 92%
ADR -> 209
HS% -> 33%
First Kill (FK) -> 1
First Death (FD) -> 0
FK - FD -> +1


In [118]:
player_stats_df = pd.DataFrame(player_stats_dict)
player_stats_df
# for key in player_stats_dict.keys():
#     print(len(player_stats_dict[key]))

,match_id,game_id,name,team,nationality,stat_scope,side,r,acs,k,d,a,k-d,kast,adr,hs%,fk,fd,fk-fd
0,102603,244560,Dambi,NS,South Korea,per_map,att,1.93,331,15,6,2,+9,92%,209,33%,1,0,+1
1,102603,244560,Xross,NS,South Korea,per_map,att,1.03,176,6,6,7,0,100%,117,15%,0,1,-1
2,102603,244560,Francis,NS,South Korea,per_map,att,0.83,233,10,11,0,-1,75%,135,24%,4,4,0
3,102603,244560,Rb,NS,South Korea,per_map,att,1.14,166,8,4,3,+4,75%,105,41%,0,1,-1
4,102603,244560,Ivy,NS,South Korea,per_map,att,0.86,102,5,5,1,0,75%,63,17%,1,0,+1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,102603,244561,BerserX,TS,Indonesia,per_map,all,1.44,263,19,12,3,+7,71%,180,16%,3,0,+3
86,102603,244561,TenTen,TS,South Korea,per_map,all,0.98,234,17,17,4,0,71%,149,15%,2,2,0
87,102603,244561,Sylvan,TS,South Korea,per_map,all,0.78,139,10,14,10,-4,76%,103,37%,0,5,-5
88,102603,244561,JessieVash,TS,Philippines,per_map,all,0.73,175,13,16,4,-3,67%,110,50%,1,1,0
